# Animer — build the anime embedding index

Runs end-to-end on Google Colab (free GPU). Total time: ~10-15 minutes.

**Before you start:**
1. Runtime → Change runtime type → **T4 GPU**
2. Have your Supabase URL + service_role key ready

**What this does:**
1. Pulls every anime from AniList (~17k)
2. Embeds synopsis + tags via BGE-base (free, runs on the Colab GPU)
3. Uploads vectors to your Supabase `anime` table

## 1. Install dependencies

In [ ]:
!pip install -q requests sentence-transformers supabase

## 2. Set credentials

Paste your Supabase values below. (Found at: Supabase dashboard → Settings → API)

In [ ]:
import os

os.environ['SUPABASE_URL'] = 'https://xxxxx.supabase.co'      # ← REPLACE
os.environ['SUPABASE_SERVICE_KEY'] = 'eyJ...'                  # ← REPLACE

# Sanity check
assert os.environ['SUPABASE_URL'].startswith('https://'), 'SUPABASE_URL not set'
assert len(os.environ['SUPABASE_SERVICE_KEY']) > 20,       'SUPABASE_SERVICE_KEY not set'
print('✓ Credentials loaded')

## 3. Fetch all anime from AniList

~5-10 minutes. Polite delay between pages so we don't get rate-limited.

In [ ]:
import json, time, requests

ANILIST_URL = 'https://graphql.anilist.co'
QUERY = '''
query ($page: Int, $perPage: Int) {
  Page(page: $page, perPage: $perPage) {
    pageInfo { hasNextPage }
    media(type: ANIME, sort: POPULARITY_DESC) {
      id idMal
      title { romaji english }
      description(asHtml: false)
      coverImage { large }
      startDate { year }
      averageScore popularity
      genres
      tags { name rank isMediaSpoiler }
      format episodes
    }
  }
}
'''

def normalize(m):
    return {
        'id': m['id'],
        'mal_id': m.get('idMal'),
        'title': m['title']['romaji'],
        'title_english': m['title'].get('english'),
        'synopsis': (m.get('description') or '').replace('<br>', ' ').strip() or None,
        'cover_url': (m.get('coverImage') or {}).get('large'),
        'year': (m.get('startDate') or {}).get('year'),
        'avg_score': m.get('averageScore'),
        'popularity': m.get('popularity'),
        'genres': m.get('genres') or [],
        'tags': [
            {'name': t['name'], 'rank': t['rank']}
            for t in (m.get('tags') or [])
            if not t.get('isMediaSpoiler')
        ],
        'format': m.get('format'),
        'episodes': m.get('episodes'),
    }

anime_list = []
page = 1
while page < 500:
    r = requests.post(ANILIST_URL,
                      json={'query': QUERY, 'variables': {'page': page, 'perPage': 50}},
                      timeout=30)
    if r.status_code == 429:
        wait = int(r.headers.get('Retry-After', 60))
        print(f'  rate-limited, sleeping {wait}s')
        time.sleep(wait); continue
    r.raise_for_status()
    data = r.json()['data']['Page']
    anime_list.extend(normalize(m) for m in data['media'])
    if page % 10 == 0:
        print(f'  page {page}: {len(anime_list)} total')
    if not data['pageInfo']['hasNextPage']:
        break
    page += 1
    time.sleep(0.7)

print(f'\n✓ Fetched {len(anime_list)} anime')

## 4. Embed every anime via BGE-base on the GPU

~3-5 minutes on T4. The model is ~430MB and downloads once.

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

model = SentenceTransformer('BAAI/bge-base-en-v1.5', device=device)

def text_for(a):
    parts = [a['title']]
    if a.get('title_english') and a['title_english'] != a['title']:
        parts.append(a['title_english'])
    if a.get('genres'):
        parts.append('Genres: ' + ', '.join(a['genres']))
    top_tags = [t['name'] for t in (a.get('tags') or []) if t['rank'] >= 60][:8]
    if top_tags:
        parts.append('Themes: ' + ', '.join(top_tags))
    if a.get('synopsis'):
        parts.append(a['synopsis'][:1500])
    return '\n'.join(parts)

texts = [text_for(a) for a in anime_list]
embeddings = model.encode(texts,
                          batch_size=64,
                          show_progress_bar=True,
                          normalize_embeddings=True,
                          convert_to_numpy=True)

print(f'\n✓ Embedded {len(embeddings)} anime, dim={embeddings.shape[1]}')

## 5. Upload to Supabase

**Important:** before running this, make sure your `anime` table's `embedding` column is `vector(768)` (BGE-base dimension). If you ran the original schema with `vector(1536)`, run this in Supabase SQL Editor first:

```sql
alter table anime alter column embedding type vector(768);
```

In [ ]:
from supabase import create_client

sb = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_SERVICE_KEY'])

BATCH = 100
for i in range(0, len(anime_list), BATCH):
    batch = anime_list[i:i+BATCH]
    vecs = embeddings[i:i+BATCH]
    rows = []
    for a, v in zip(batch, vecs):
        rows.append({
            'id': a['id'],
            'mal_id': a.get('mal_id'),
            'title': a['title'],
            'title_english': a.get('title_english'),
            'synopsis': a.get('synopsis'),
            'cover_url': a.get('cover_url'),
            'year': a.get('year'),
            'avg_score': a.get('avg_score'),
            'popularity': a.get('popularity'),
            'genres': a.get('genres') or [],
            'tags': a.get('tags') or [],
            'format': a.get('format'),
            'episodes': a.get('episodes'),
            'embedding': v.tolist(),
        })
    sb.table('anime').upsert(rows, on_conflict='id').execute()
    print(f'  uploaded {min(i+BATCH, len(anime_list))}/{len(anime_list)}')

print('\n✓ All done!')

## 6. Quick sanity check

Verify the data is queryable.

In [ ]:
result = sb.table('anime').select('id,title,year,avg_score').limit(5).execute()
for row in result.data:
    print(row)